In [11]:
# ==========================================
# سلول ۱: اتصال به درایو و نصب تمام پیش‌نیازها
# ==========================================
from google.colab import drive
import os

# 1. اتصال به درایو
drive.mount('/content/drive')

# 2. رفتن به پوشه پروژه در درایو
os.chdir('/content/drive/MyDrive/ai_room_project')

print("⏳ در حال نصب کتابخانه‌های پایتون (ممکن است چند دقیقه طول بکشد)...")
!pip install -q -r requirements.txt
!pip install -q sacremoses sentencepiece

print("⏳ در حال نصب و اجرای دیتابیس Redis...")
!apt-get update -qq
!apt-get install redis-server -y -qq
!service redis-server start

print("⏳ در حال نصب ابزار تونل (Cloudflared)...")
!curl -sL --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

# ساخت پوشه‌های ضروری عکس
os.makedirs("static/uploads", exist_ok=True)
os.makedirs("static/generated", exist_ok=True)

print("\n✅ مرحله اول با موفقیت تمام شد! حالا سلول دوم را اجرا کنید.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ در حال نصب کتابخانه‌های پایتون (ممکن است چند دقیقه طول بکشد)...
⏳ در حال نصب و اجرای دیتابیس Redis...
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Starting redis-server: redis-server.
⏳ در حال نصب ابزار تونل (Cloudflared)...
(Reading database ... 127000 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.9.1) over (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...

✅ مرحله اول با موفقیت تمام شد! 

In [12]:
# ==========================================
# سلول ۲: اجرای سرور هوش مصنوعی و تونل اینترنت
# ==========================================
import subprocess
import time
import re
import os

# 1. پاکسازی پروسه‌های گیر کرده از اجراهای قبلی
os.system("pkill -f uvicorn")
os.system("pkill -f cloudflared")
time.sleep(2)

print("⏳ در حال اجرای سرور FastAPI در پس‌زمینه...")
# 2. اجرای سرور (لاگ‌ها در فایل uvicorn.log ذخیره می‌شوند)
subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
                 stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6) # صبر می‌کنیم تا سرور بالا بیاید

print("⏳ در حال ساخت تونل کلودفلر...")
# 3. اجرای تونل برای اتصال به اینترنت
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12) # صبر برای تخصیص لینک از سمت کلودفلر

# 4. پیدا کردن لینک عمومی از داخل لاگ‌ها
tunnel_url = None
with open("tunnel.log", "r") as f:
    content = f.read()
    match = re.search(r"https?://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if match:
        tunnel_url = match.group(0)

if tunnel_url:
    print("\n" + "⭐"*25)
    print("✅ سرور و تونل با موفقیت اجرا شدند!")
    print("🚀 لینک جدید سرور شما:")
    print(f"👉  {tunnel_url}  👈")
    print("⭐"*25 + "\n")
    print("⚠️ توجه: این تب مرورگر کولب را باز نگه دارید تا سرور خاموش نشود.")
else:
    print("❌ پیدا کردن لینک با مشکل مواجه شد. لاگ‌ها:")
    print(open("tunnel.log", "r").read())

⏳ در حال اجرای سرور FastAPI در پس‌زمینه...
⏳ در حال ساخت تونل کلودفلر...

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
✅ سرور و تونل با موفقیت اجرا شدند!
🚀 لینک جدید سرور شما:
👉  https://ext-hill-rome-routines.trycloudflare.com  👈
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐

⚠️ توجه: این تب مرورگر کولب را باز نگه دارید تا سرور خاموش نشود.
